In [ ]:
# Export model for deployment
export_dir = PROJECT_ROOT / "results" / "models"
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Save as PyTorch state dict (for loading in Python)
state_dict_path = export_dir / "ast_model_state_dict.pth"
torch.save(model.state_dict(), state_dict_path)
print(f"✅ Model state dict saved to {state_dict_path}")

# 2. Save complete model with metadata
model_metadata = {
    'model_state_dict': model.state_dict(),
    'config': config,
    'class_names': class_names,
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1)
    }
}
complete_model_path = export_dir / "ast_model_complete.pth"
torch.save(model_metadata, complete_model_path)
print(f"✅ Complete model saved to {complete_model_path}")

# 3. Export to TorchScript (for C++ deployment)
try:
    dummy_input = torch.randn(1, 1, config['preprocessing']['n_mels'], 1000).to(device)
    traced_model = torch.jit.trace(model, dummy_input)
    torchscript_path = export_dir / "ast_model.pt"
    traced_model.save(str(torchscript_path))
    print(f"✅ TorchScript model saved to {torchscript_path}")
except Exception as e:
    print(f"⚠️  TorchScript export failed: {e}")

# Summary
print(f"\n{'='*60}")
print("✅ Model Training and Evaluation Complete!")
print(f"{'='*60}")
print(f"\nResults Summary:")
print(f"  - Test Accuracy:  {accuracy:.4f}")
print(f"  - Test Precision: {precision:.4f}")
print(f"  - Test Recall:    {recall:.4f}")
print(f"  - Test F1-Score:  {f1:.4f}")
print(f"\nSaved Outputs:")
print(f"  - Best checkpoint: {checkpoint_dir / 'best_model.pth'}")
print(f"  - Visualizations: {PROJECT_ROOT / 'results' / 'figures'}/")
print(f"  - Exported models: {export_dir}/")
print(f"{'='*60}")

## Section 7: Model Export for Deployment

In [ ]:
# Inference examples
def predict_sample(model, spec, device):
    """Predict on a single spectrogram"""
    model.eval()
    with torch.no_grad():
        spec_tensor = torch.FloatTensor(spec).unsqueeze(0).unsqueeze(0).to(device)
        output = model(spec_tensor)
        probs = F.softmax(output, dim=1).cpu().numpy()[0]
        pred_class = output.argmax(dim=1).item()
    return pred_class, probs

# Get random test samples to visualize
np.random.seed(42)
sample_indices = np.random.choice(len(test_dataset), 5, replace=False)

fig, axes = plt.subplots(5, 2, figsize=(14, 12))

for idx, sample_idx in enumerate(sample_indices):
    spec, label = test_dataset[sample_idx]
    spec_np = spec.squeeze().numpy()
    
    # Prediction
    pred_class, probs = predict_sample(model, spec_np, device)
    
    # Plot spectrogram
    im = axes[idx, 0].imshow(spec_np, cmap='viridis', aspect='auto', origin='lower')
    axes[idx, 0].set_title(f'True: {class_names[label]}, Pred: {class_names[pred_class]}')
    axes[idx, 0].set_ylabel('Frequency')
    plt.colorbar(im, ax=axes[idx, 0])
    
    # Plot probabilities
    top_indices = np.argsort(probs)[-5:][::-1]
    axes[idx, 1].barh([class_names[i] for i in top_indices], probs[top_indices])
    axes[idx, 1].set_xlabel('Probability')
    axes[idx, 1].set_xlim([0, 1])

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'inference_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Inference examples visualization saved")

## Section 6: Model Inference on Test Samples

In [ ]:
# Compute per-class metrics
precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
recall_per_class = recall_score(all_labels, all_preds, average=None, zero_division=0)
f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

# Create DataFrame
metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class
})

print("\n✅ Per-Class Metrics:")
print(metrics_df.to_string(index=False))

# Plot per-class F1-scores
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(f1_per_class)), f1_per_class, color='steelblue', alpha=0.7)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('Per-Class F1-Scores', fontsize=14)
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, f1_per_class)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}', 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Per-class metrics visualization saved")

## Section 5: Per-Class Performance Analysis

In [ ]:
# Plot ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_true_onehot = np.eye(num_classes)[all_labels]

# ROC curves
for i in range(min(5, num_classes)):  # Plot first 5 classes
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{class_names[i]} (AUC={roc_auc:.2f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves (First 5 Classes)', fontsize=14)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# PR curves
for i in range(min(5, num_classes)):
    precision_curve, recall_curve, _ = precision_recall_curve(y_true_onehot[:, i], all_probs[:, i])
    f1_pr = np.nanmean([2 * (p * r) / (p + r + 1e-10) for p, r in zip(precision_curve, recall_curve)])
    axes[1].plot(recall_curve, precision_curve, label=f'{class_names[i]} (F1={f1_pr:.2f})', linewidth=2)

axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves (First 5 Classes)', fontsize=14)
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ ROC and PR curves saved")

## Section 4: ROC and Precision-Recall Curves

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=14)
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', ax=axes[1], cbar_kws={'label': 'Normalized'})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14)
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved")

## Section 3: Confusion Matrix

In [ ]:
# Load test dataset and model (copy of Dataset class from notebook 02)
class AudioSpectrogramDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.files = sorted(self.data_dir.glob("*.npy"))
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        spec = np.load(self.files[idx])
        filename = self.files[idx].stem
        label = int(filename.split('_')[0])
        spec_tensor = torch.FloatTensor(spec).unsqueeze(0)
        label_tensor = torch.LongTensor([label])[0]
        return spec_tensor, label_tensor

# Load test dataset
test_dataset = AudioSpectrogramDataset(PROJECT_ROOT / "data" / "processed" / "test")
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Load best model checkpoint
checkpoint_path = PROJECT_ROOT / "results" / "checkpoints" / "best_model.pth"

# Recreate model (copy PatchEmbedding, MultiHeadAttention, TransformerBlock, AudioSpectrogramTransformer from notebook 02)
class PatchEmbedding(nn.Module):
    def __init__(self, input_shape, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return self.dropout(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class AudioSpectrogramTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.patch_embed = PatchEmbedding(config['input_shape'], config['patch_size'], config['embed_dim'])
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, config['embed_dim']))
        self.blocks = nn.ModuleList([
            TransformerBlock(config['embed_dim'], config['num_heads'], config.get('mlp_ratio', 4.0), config.get('dropout', 0.1))
            for _ in range(config['num_layers'])
        ])
        self.norm = nn.LayerNorm(config['embed_dim'])
        self.head = nn.Linear(config['embed_dim'], config['num_classes'])
        self._init_weights()
    
    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_module_weights)
    
    def _init_module_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.head(x)
        return x

# Build and load model
model_config = config['model'].copy()
model_config['input_shape'] = [config['preprocessing']['n_mels'], 1000]
model = AudioSpectrogramTransformer(model_config)

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Model loaded from {checkpoint_path}")
    print(f"   Epoch: {checkpoint['epoch']}")
    print(f"   Validation Accuracy: {checkpoint['val_acc']:.2f}%")
else:
    print(f"⚠️  Checkpoint not found at {checkpoint_path}")

model = model.to(device)
model.eval()

# Evaluate on test set
all_preds = []
all_labels = []
all_probs = []

print("\nEvaluating on test set...")
with torch.no_grad():
    for specs, labels in tqdm(test_loader):
        specs = specs.to(device)
        labels = labels.cpu().numpy()
        
        outputs = model(specs)
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels)
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Compute metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"\n✅ Test Set Metrics:")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")

## Section 2: Evaluate on Test Set

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score, accuracy_score, precision_score, recall_score
)
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Setup
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# Load config
with open(PROJECT_ROOT / "configs" / "config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Define class names (from config or default)
class_names = config.get('class_names', [f'Class_{i}' for i in range(config['model']['num_classes'])])
num_classes = config['model']['num_classes']

print(f"✅ Configuration loaded")
print(f"   Number of classes: {num_classes}")
print(f"   Classes: {', '.join(class_names[:5])}{'...' if len(class_names) > 5 else ''}")

## Section 1: Setup and Load Model

# Audio Event Detection - Results & Visualization
## Model Evaluation and Performance Analysis

This notebook provides:
1. Model evaluation on test set
2. Comprehensive metrics computation
3. Confusion matrix visualization
4. ROC and Precision-Recall curves
5. Per-class performance analysis
6. Model inference on new samples
7. Model export for deployment

In [ ]:
# Placeholder for 03_results_visualization.ipynb
{}